# HarnessProof Mini — Cloud GPU Lab

Browser-only fine-tuning experiment. Run on **Colab Free or Kaggle GPU**. The student is Qwen2.5 0.5B; DeepSeek/OpenCode generates and audits training examples in a **separate browser Codespaces phase**, subject to a genuinely free API route. No training occurs on the Mac. GPUs are free-tier limited and not guaranteed.

**Before running:** Publish the project source to your public GitHub repo, replace `REPO_URL` below with its real URL, and enable a GPU. Nothing uploads model weights or metrics to external services automatically.


In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path
try:
    import torch
    assert torch.cuda.is_available(), 'GPU unavailable: enable the free GPU in notebook runtime settings'
    print('Free-tier GPU detected:', torch.cuda.get_device_name(0))
except Exception as exc:
    raise RuntimeError('This experiment requires a free NVIDIA GPU runtime; do not switch to paid hardware.') from exc


In [ ]:
# CHANGE this to YOUR public GitHub repository. It must contain the delivered HarnessProof files.
REPO_URL = "https://github.com/YOUR_ACCOUNT/harnessproof-mini.git"
if 'YOUR_ACCOUNT' in REPO_URL or not REPO_URL.startswith('https://github.com/'):
    raise ValueError('Set REPO_URL to your public GitHub repo first.')
base_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
project_dir = base_dir / 'harnessproof-mini'
if not project_dir.exists():
    subprocess.check_call(['git','clone','--depth','1',REPO_URL,str(project_dir)])
os.chdir(project_dir)
print('Project:', project_dir)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements-cloud.txt'])
import trl, transformers, peft, datasets
print('Versions:', {'torch':torch.__version__, 'transformers':transformers.__version__, 'trl':trl.__version__, 'peft':peft.__version__, 'datasets':datasets.__version__})
subprocess.check_call([sys.executable,'-m','harnessproof.cli','build-data'])
subprocess.check_call([sys.executable,'-m','scripts.quality_gate','--mode','pilot'])


## Run 1 — QLoRA (4-bit frozen base + LoRA adapter)

This **runs the base-model evaluation, trains an adapter, then evaluates it on the same held-out set**. A 40-step pilot is not a production model.


In [ ]:
subprocess.check_call([sys.executable,'-m','scripts.cloud_train','--mode','qlora','--steps','40','--eval-max','18'])
summary_path=Path('runs/cloud/qlora/summary.json')
print(summary_path.read_text())


## Run 2 — Optional LoRA; Run 3 — Optional full-layer baseline

Turn these on only when free GPU quota and memory permit. Full-layer mode does not auto-save giant weights.


In [ ]:
RUN_LORA = False
RUN_FULL = False
if RUN_LORA:
    subprocess.check_call([sys.executable,'-m','scripts.cloud_train','--mode','lora','--steps','40','--eval-max','18'])
if RUN_FULL:
    subprocess.check_call([sys.executable,'-m','scripts.cloud_train','--mode','full','--steps','40','--eval-max','18'])


## Agent tool test and tokenizer context budget

This demonstration **may fail if the trained model emits malformed JSON**—that is valid measured failure evidence. The four roles are router, retriever, executor and verifier, not four separate LLMs.


In [ ]:
subprocess.run([sys.executable,'-m','scripts.cloud_harness','--quantize',
                '--adapter-path','runs/cloud/qlora/adapter',
                '--request','Find the prompt caching documentation'],check=False)
subprocess.run([sys.executable,'-m','scripts.context_budget','--budget','512'],check=False)


## Preserve lightweight results in cloud output

Kaggle: successful version runs retain files in `/kaggle/working`. Colab: runtime storage is ephemeral; optionally copy summaries to Drive or another approved cloud destination. The code below copies **metrics only** to the notebook top-level output directory; it never publishes weights.


In [ ]:
for mode in ('qlora','lora','full'):
    source=Path(f'runs/cloud/{mode}/summary.json')
    if source.exists():
        target=base_dir / f'harnessproof_{mode}_summary.json'
        shutil.copyfile(source,target)
        print('Cloud output:',target)
print('Experiment notebook finished; publishing results and adapters remains a separate authorized action.')
